# ML-10 — Content Action Playbook

The queue: what to do first, and why, in words a human trusts.

> How to run: mount Drive, paste your HF READ token when prompted, then Runtime → Run all. First full build takes ~5 minutes (RF train + two warehouse pulls); later runs load the cached scored matrix from Drive.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Data discovery & debugging trail (kept for transparency)

> **Note on process:** the taxonomy in §1 was not designed in a vacuum. It emerged from diagnostic passes on real data; each failed attempt taught us something about the data's actual structure. This section keeps the one diagnostic that changed the design running live, and narrates the rest.

### Critical finding: namespace mismatch between datasets

`content_refresh_anonymized.csv` (starter CSV from the research paper) and `internship-warehouse` (daily performance facts) use **different anonymization namespaces** (12-char vs 16-char hex suffixes). Verified live in cell 0.3: **0% ID overlap**.

**Implication:** metadata that lives only in the starter CSV (`word_count`, `days_since_last_update`) cannot be joined to warehouse features. This is a structural property of two separate datasets, not a merge bug.

**Consequences for this notebook:**
- `thin_with_potential` (word_count-based) was **removed** from the taxonomy.
- `content_age_days` is computed from `MIN(report_date)` in the warehouse — more honest than snapshot age (no survivor bias).
- Trend-based signals needing 30-day lags are **defined but not activated** in the 10-day test window (see the WATCH note in §1).

### What we learned through iteration (outputs quoted; attempt cells superseded)

1. First taxonomy (5 design-doc codes) → **94%** of queued rows `signal_unclear`.
2. Second pass (age from warehouse, 4 snapshot codes) → **63.4%** unclear.
3. Diagnostic on queued rows showed the RF never selects zero-click pages (0 of 500) and 100% of queued rows carry GA4 data → taxonomy rebuilt around observed model behavior.
4. Final taxonomy → **0.0%** unclear.

Superseded attempt cells were removed; their outputs are quoted above and preserved in repo history. The live diagnostic below is the one that matters.

In [ ]:
import os, getpass
import numpy as np
import pandas as pd
import duckdb
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/FlyRankai-Internship/work/outputs'
os.makedirs(DRIVE_BASE, exist_ok=True)
SCORED_PATH = f'{DRIVE_BASE}/w07_scored_matrix.parquet'
CLEAN_PATH = f'{DRIVE_BASE}/clean_features_label.parquet'
FORCE_REBUILD = False  # set True to retrain RF and re-pull from warehouse

con = duckdb.connect()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

RANDOM_SEED = 42
ACT_K, REVIEW_K = 20, 50
print('Setup OK. DRIVE_BASE =', DRIVE_BASE)

In [ ]:
# --- warehouse pulls: windows end at decision date; past only, never future ---
def pull_impressions(con, rel):
    return con.sql('''
        SELECT content_hash_id, report_date, gsc_impressions
        FROM read_parquet(\'''' + rel + '''/fact_content_daily_performance/**/*.parquet\')
        WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
    ''').df()

def pull_age(con, rel, pages_df):
    con.register('matrix_pages', pages_df[['content_hash_id']].drop_duplicates())
    out = con.sql('''
        SELECT f.content_hash_id,
               DATE '2026-03-31' - MIN(f.report_date) AS content_age_days
        FROM read_parquet(\'''' + rel + '''/fact_content_daily_performance/**/*.parquet\') f
        WHERE f.content_hash_id IN (SELECT content_hash_id FROM matrix_pages)
        GROUP BY f.content_hash_id
    ''').df()
    con.unregister('matrix_pages')
    return out

if os.path.exists(SCORED_PATH) and not FORCE_REBUILD:
    m = pd.read_parquet(SCORED_PATH)
    print(f'Loaded cached scored matrix: {len(m):,} rows')
    if 'content_age_days' not in m.columns:
        m = m.merge(pull_age(con, REL, m), on='content_hash_id', how='left')
        print('Backfilled content_age_days from warehouse')
else:
    from sklearn.ensemble import RandomForestClassifier
    clean = pd.read_parquet(CLEAN_PATH)
    clean['report_date'] = pd.to_datetime(clean['report_date'])
    m = clean.merge(pull_impressions(con, REL), on=['content_hash_id', 'report_date'], how='left', validate='one_to_one')
    m = m.merge(pull_age(con, REL, m), on='content_hash_id', how='left')

    FEATURES = ['gsc_clicks', 'gsc_avg_position', 'has_ga4_data', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'gsc_avg_position_is_placeholder']
    SKEWED = ['gsc_clicks', 'gsc_avg_position', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic']
    def prep(X):
        Xp = X[FEATURES].copy()
        for c in SKEWED:
            Xp[c] = np.log1p(Xp[c].fillna(0))
        return Xp

    train_mask = m['report_date'] < '2026-03-22'
    test_mask = m['report_date'] >= '2026-03-22'
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
    rf.fit(prep(m[train_mask]), m[train_mask]['recovery_label'])
    m.loc[test_mask, 'rf_score'] = rf.predict_proba(prep(m[test_mask]))[:, 1]
    m.loc[train_mask, 'rf_score'] = np.nan

    IMPRESSION_THRESHOLD = 194  # LOCKED in W04 §0.2 - observed P90 of March 2026 impressions; do not re-derive here
    gate = (m['gsc_avg_position'].between(1, 10) & (m['gsc_impressions'] >= IMPRESSION_THRESHOLD) & (m['gsc_clicks'] == 0))
    m['rule_score'] = np.where(gate, m['gsc_impressions'], 0)

    m.to_parquet(SCORED_PATH, index=False)
    print(f'Built and cached scored matrix: {len(m):,} rows -> {SCORED_PATH}')

m['report_date'] = pd.to_datetime(m['report_date'])
REQUIRED = ['rf_score', 'rule_score', 'gsc_impressions', 'content_age_days', 'recovery_label']
missing = [c for c in REQUIRED if c not in m.columns]
assert not missing, f'scored matrix missing {missing}; set FORCE_REBUILD = True and rerun'
print(f'Ready: {len(m):,} rows | test-window rows: {(m["report_date"] >= "2026-03-22").sum():,}')

In [ ]:
# === 0.3 Live diagnostic: starter CSV vs warehouse namespace overlap ===
starter = pd.read_csv('https://raw.githubusercontent.com/ziadzakaryaai-ux/FlyRankai-Internship/main/data/raw/content_refresh_anonymized.csv')
starter_ids = set(starter['content_id'].astype(str))
matrix_ids = set(m['content_hash_id'].astype(str))
overlap = starter_ids & matrix_ids
print(f'Unique pages in scored matrix: {len(matrix_ids):,}')
print(f'Unique pages in starter CSV:   {len(starter_ids):,}')
print(f'Overlap: {len(overlap):,} ({len(overlap) / len(matrix_ids):.1%})')
print('=> starter-CSV metadata (word_count, days_since_last_update) is NOT joinable;')
print('   the taxonomy uses warehouse-derived signals only (see §0 narrative).')

## 1. Ranked actions + reason codes

### Design decisions

1. **The score is a ranking device, not a decision.** The RF probability orders pages by predicted recovery probability; it does not tell the specialist what to do. A separate deterministic layer maps observable signals to actionable reason codes.
2. **Why rule-based codes, not SHAP:** the locked feature set includes instrumentation artifacts (`has_ga4_data`, `gsc_avg_position_is_placeholder`). SHAP on these would tell a human 'because no GA4' — reading a measurement gap as a content problem. Rule-based codes live independently of model internals and survive retraining.
3. **Why fixed categories, not continuous ranges:** capacity is discrete (~20–50 pages/week), tools are discrete, and auditability requires categorical dispositions for §4.

### Tiers

| Tier | Population | Capacity |
|---|---|---|
| `ACT_THIS_WEEK` | top 20 by RF score per date | reserved |
| `REVIEW_IF_CAPACITY` | ranks 21–50 per date | if bandwidth allows |
| `WATCH` | rank >50 AND `trend_30d <= -40%` AND `impressions >= demand_median` | zero capacity; glance list |
| `NO_ACTION_LOGGED` | everything else | explicit silent state |

> **WATCH note:** `trend_30d` needs a 30-day lag the 10-day test window cannot provide, so WATCH is **defined but yields 0 rows here**; it activates in production runs where ≥30 days of history precede the decision date. Stated here so the table and the code agree.

### Reason codes (deterministic, fixed priority, first fire = primary)

| Code | Condition | Plain-language reason |
|---|---|---|
| `stale_visible` | age ≥ 180d AND impressions ≥ same-date median | Old page, still earning traffic — refresh candidate |
| `strong_engagement` | has_ga4_data AND sessions_organic ≥ same-date 75th pct | Users engage deeply — protect from decay |
| `high_position_traffic` | position < 5 AND impressions ≥ same-date 75th pct | Top position, high visibility — expand opportunity |
| `converting_visibility` | page-one AND clicks > 0 AND impressions ≥ median | Ranking and converting — maintain |
| `signal_unclear` | fallback | Model flagged, observable signals do not explain |

In [ ]:
test_df = m[m['report_date'] >= '2026-03-22'].copy()

# --- tiers ---
test_df['rank_rf'] = test_df.groupby('report_date')['rf_score'].rank(ascending=False, method='first')
test_df['tier'] = 'NO_ACTION_LOGGED'
test_df.loc[test_df['rank_rf'] <= REVIEW_K, 'tier'] = 'REVIEW_IF_CAPACITY'
test_df.loc[test_df['rank_rf'] <= ACT_K, 'tier'] = 'ACT_THIS_WEEK'

# WATCH defined but not activated in this window (no 30-day lag available)
test_df['trend_30d'] = np.nan
print('WATCH tier defined; not activated in this window (no 30-day lag available).')

# --- cohort thresholds: cross-section of the SAME report_date (known at d) ---
test_df['demand_median_d'] = test_df.groupby('report_date')['gsc_impressions'].transform('median')
test_df['sessions_q75_d'] = test_df.groupby('report_date')['sessions_organic'].transform(lambda s: s.quantile(0.75))
test_df['impr_q75_d'] = test_df.groupby('report_date')['gsc_impressions'].transform(lambda s: s.quantile(0.75))
high_demand = test_df['gsc_impressions'] >= test_df['demand_median_d']
page_one = test_df['gsc_avg_position'].between(1, 10)

# --- reason codes ---
conds = [
    (test_df['content_age_days'] >= 180) & high_demand,
    (test_df['has_ga4_data'] == 1) & (test_df['sessions_organic'] >= test_df['sessions_q75_d']),
    (test_df['gsc_avg_position'] < 5) & (test_df['gsc_impressions'] >= test_df['impr_q75_d']),
    page_one & (test_df['gsc_clicks'] > 0) & high_demand,
]
codes = ['stale_visible', 'strong_engagement', 'high_position_traffic', 'converting_visibility']
queued = test_df['tier'].isin(['ACT_THIS_WEEK', 'REVIEW_IF_CAPACITY'])
test_df['reason_code'] = np.where(queued, np.select(conds, codes, default='signal_unclear'), np.nan)

# --- cross-disagree flag (RF vs rule) ---
IMPRESSION_THRESHOLD = 194  # LOCKED in W04 §0.2 - observed P90 of March 2026 impressions; do not re-derive here
gate = (page_one & (test_df['gsc_impressions'] >= IMPRESSION_THRESHOLD) & (test_df['gsc_clicks'] == 0))
test_df['rule_score'] = np.where(gate, test_df['gsc_impressions'], 0)
test_df['rank_rule'] = test_df.groupby('report_date')['rule_score'].rank(ascending=False, method='first')
test_df['cross_disagree'] = (
    ((test_df['rank_rf'] <= ACT_K) & (test_df['rank_rule'] > REVIEW_K)) |
    ((test_df['rank_rule'] <= ACT_K) & (test_df['rank_rf'] > REVIEW_K))
)

queued_df = test_df[queued].copy()
print('=== Tier distribution (test window) ===')
print(test_df['tier'].value_counts().to_string())
print('\n=== Reason code distribution (queued) ===')
print(queued_df['reason_code'].value_counts().to_string())
print(f'\nsignal_unclear share: {(queued_df["reason_code"] == "signal_unclear").mean():.1%}')
print(f'queued rows: {len(queued_df):,}')

In [ ]:
# === Disagreement anatomy: where does cross_disagree fire? ===
clause1 = (test_df['rank_rf'] <= ACT_K) & (test_df['rank_rule'] > REVIEW_K)
clause2 = (test_df['rank_rule'] <= ACT_K) & (test_df['rank_rf'] > REVIEW_K)
print('=== Window-level ===')
print(f'total flagged: {test_df["cross_disagree"].sum():,}')
print(f'  clause 1 (RF top-20 the rule rejects): {clause1.sum():,}')
print(f'  clause 2 (rule top-20 the RF rejects): {clause2.sum():,}')
print('\n=== Queue-level ===')
print(f'queued flagged: {queued_df["cross_disagree"].sum()} / {len(queued_df)} ({queued_df["cross_disagree"].mean():.0%})')
print(queued_df.groupby('tier')['cross_disagree'].agg(flagged='sum', n='size', share='mean').to_string())
print('\n=== Disjointness (the cleaner statistic) ===')
print(f'queued rows also in rule top-50 on same date: {(queued_df["rank_rule"] <= REVIEW_K).sum()} / {len(queued_df)}')
print('\n=== Concentration by reason code ===')
print(queued_df.groupby('reason_code')['cross_disagree'].agg(flagged='sum', n='size', share='mean').to_string())

### Headline finding: the two triage systems share zero pages — quantified

Measured on the Mar 22–31 test window (10 decision dates); numbers printed by the anatomy cell above:

| Quantity | Value |
|---|---|
| Queued rows (RF top-50 per date) | 500 |
| Queued rows also in the rule's top-50 on the same date | **0** |
| `cross_disagree` flags, whole window | 400 |
| — clause 1: RF top-20 the rule rejects | 200 (= 100% of `ACT_THIS_WEEK`) |
| — clause 2: rule top-20 the RF rejects | 200 (all outside the queue by construction) |
| Flagged share within the queue | 200/500 (40%), concentrated entirely in `ACT_THIS_WEEK`; structurally 0 in `REVIEW_IF_CAPACITY` |

**Correction to an earlier framing (stated, not silently fixed):** an earlier draft described this as '80% of queued rows disagree'. That was a misread of a window-level count (400) as a queue-level share. The queue-level figure is 40% flagged, and the unflagged 300 are unflagged *by construction* (the flag only compares each system's top-20 against the other's outside-50). The stronger and simpler statement of disjointness is the zero overlap: on no date do the two systems' top-50 lists share a single row.

**Interpretation:** population disjointness, not per-row noise. The rule targets zero-click, high-visibility pages (CTR problems); the RF consistently selects GA4-tracked, engagement-rich pages (0 of 500 queued rows are zero-click). Every `ACT_THIS_WEEK` row is a page the existing workflow would never have surfaced — which is why §3 routes all `cross_disagree` rows to mandatory human adjudication.

**Limitation of the flag as designed:** it cannot fire on ranks 21–50, so it is a population-boundary marker, not a per-row agreement measure. For per-row agreement across the full queue, compare `rank_rule` bands directly.

**Connection to ML-09 Finding #4:** consistent with refresh acting as a *stabilization brake* for already-valuable pages, not a recovery lever for broken ones.

## 2. Intended use and limits

**Who:** the SEO specialist at FlyRank, for the weekly triage meeting. **For what:** prioritizing a limited review budget (~20–50 pages/week). The system surfaces candidates and proposes a diagnostic framing; it does not prescribe the fix.

**Where it stops being valid:**
1. Trained on Jan–Mar 2026; beyond June 2026 without retraining, predictions are extrapolation.
2. Pages younger than 30 days: insufficient history; they default to `signal_unclear`.
3. Post-core-update weeks: base rate shifts materially (ML-09 observed 0.554 → 0.476); decisions in the 14 days after a confirmed update are directional only.
4. Non-GSC-tracked pages are structurally excluded.
5. YMYL content is routed to mandatory manual review regardless of score (§3).

**What it explicitly does not do:** no auto-publish/delete/edit; no causal claim that refresh causes recovery; no prediction of Google's algorithm — only the measured recovery label.

In [ ]:
# === Limit checks on the live queue ===
ga4_share = (queued_df['has_ga4_data'] == 1).mean()
new_share = (queued_df['content_age_days'] < 30).mean()
print(f'Queued rows with GA4 instrumentation: {ga4_share:.1%}')
print(f'Queued rows younger than 30 days: {new_share:.1%}')
print('Dependency note: queue composition depends on GA4 coverage; if instrumentation drops, the queue shifts. Monitored in §4.')

## 3. Human review + the no-go list

**Mandatory manual review before any action:**
1. Brand / legal sensitivity — legal sign-off for trademark, regulatory, compliance content.
2. Recent edits (< 7 days) — skip; attribute change to the edit, not the queue.
3. `cross_disagree == True` — two legitimate triage systems disagree; human adjudicates.
4. `reason_code == signal_unclear` — investigate first; may indicate drift.

**No-go list (never automate):** auto-publish / auto-delete; YMYL edits without sign-off; edits on pages with < 7 days of observation; bulk action on `signal_unclear` rows (a drift canary, not a disposition).

**The four dispositions the specialist records:** `refresh`, `rewrite_or_consolidate`, `monitor`, `no_action_with_note` (the last one is data for §4).

In [ ]:
queued_df['needs_manual_review'] = (queued_df['cross_disagree']) | (queued_df['reason_code'] == 'signal_unclear')
print(f'Rows routed to mandatory human adjudication: {queued_df["needs_manual_review"].sum()} / {len(queued_df)}')
print('No-go list is enforced by process, not code: this tool never writes to the CMS.')

## 4. Monitoring / retrain triggers

| Trigger | Threshold | Action |
|---|---|---|
| Base-rate drift | recovery mean < 0.45 for 2 consecutive weeks | retrain / recalibrate |
| Top-K degradation | Precision@20 on live sample < 0.70 for 3 consecutive days | audit feature pipeline |
| `signal_unclear` explosion | > 40% of queued rows for 1 week | taxonomy gap or drift |
| Google core update | confirmed update | 14-day moratorium on queue decisions |
| Instrumentation drift | `has_ga4_data` or placeholder share shifts > 5pp week-over-week | audit data pipeline, not model |

**Success criteria (measured, not guaranteed):** for the `ACT_THIS_WEEK` cohort, recovery rate ≥ base rate + 15pp, and `refresh` dispositions recover more often than `no_action_with_note`. If both fail for 4 consecutive weeks, the playbook is retired or redesigned.

In [ ]:
test_df['iso_week'] = test_df['report_date'].dt.isocalendar().week.astype(int)
print('=== Base rate by ISO week (test window) ===')
print(test_df.groupby('iso_week')['recovery_label'].mean().round(3).to_string())
print('\n=== Monitoring thresholds (versioned in-repo via this cell) ===')
THRESHOLDS = {
    'base_rate_drift': 'recovery mean < 0.45 for 2 consecutive weeks',
    'topk_degradation': 'Precision@20 live < 0.70 for 3 consecutive days',
    'signal_unclear_explosion': '> 40% of queued rows for 1 week',
    'core_update': 'confirmed update -> 14-day moratorium',
    'instrumentation_drift': 'ga4/placeholder share shifts > 5pp week-over-week',
}
for k, v in THRESHOLDS.items():
    print(f'- {k}: {v}')

## 5. Exports for the paper

Exactly three artifacts are written, all to the Drive-backed `work/outputs/` (never to the ephemeral Colab filesystem). The markdown table and the code below agree by construction:

| File | Content | Feeds |
|---|---|---|
| `w07_action_queue.csv` | top-50 queue with reason codes, `cross_disagree`, `needs_manual_review` | paper §6 |
| `w07_tier_summary.csv` | tier population counts | paper §6 table |
| `w07_scored_matrix.parquet` | full scored test window (rf_score, rule_score, recovery_label) | paper appendix; doubles as the predictions artifact the capstone verifies metrics from live |

**Paper-safe language:** observed, measured, directional, associated-with, decision-support. No causal verbs.

In [ ]:
QUEUE_PATH = f'{DRIVE_BASE}/w07_action_queue.csv'
TIER_PATH = f'{DRIVE_BASE}/w07_tier_summary.csv'

queue_export = queued_df[[
    'content_hash_id', 'report_date', 'tier', 'reason_code', 'rf_score', 'rank_rf',
    'gsc_impressions', 'gsc_avg_position', 'gsc_clicks', 'sessions_organic',
    'content_age_days', 'cross_disagree', 'needs_manual_review',
]].sort_values(['report_date', 'rank_rf'])
queue_export.to_csv(QUEUE_PATH, index=False)

tier_summary = test_df['tier'].value_counts().rename_axis('tier').reset_index(name='count')
tier_summary.to_csv(TIER_PATH, index=False)

print(f'wrote {len(queue_export):,} rows -> {QUEUE_PATH}')
print(f'wrote {len(tier_summary)} rows  -> {TIER_PATH}')
print(f'scored matrix cached        -> {SCORED_PATH}')
print('3 artifacts total; matches the §5 markdown table exactly.')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.